In [1]:
import gradio as gr
print(gr.__version__)

6.0.2


In [2]:
import gradio as gr
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

model_path = "C:/Users/Parth Pathak/OneDrive - University of Waterloo/CS 679/finbert_chatbot/model"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)

labels = ["positive", "negative", "neutral"]


def classify_sentiment(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    outputs = model(**inputs)
    probs = torch.softmax(outputs.logits, dim=1).detach().numpy()[0]

    pred = labels[probs.argmax()]

    # Return TWO outputs:
    # 1) Predicted label (string)
    # 2) Probabilities for Label component (dict of floats)
    prob_dict = {
        "positive": float(probs[0]),
        "negative": float(probs[1]),
        "neutral": float(probs[2])
    }

    return pred, prob_dict


""""iface = gr.Interface(
    fn=classify_sentiment,
    inputs=gr.Textbox(label="Enter financial text"),
    outputs=[
        gr.Textbox(label="Predicted Sentiment"),
        gr.Label(label="Probability Scores")
    ],
    title="FinBERT Financial Sentiment Classifier"
)
#iface.launch()"""

# -------------------------
# Dashboard UI 
# -------------------------
with gr.Blocks() as demo:

    # Force Light Mode
    gr.HTML("""
    <style>
        body, .gradio-container {
            background-color: #ffffff !important;
            color: #000000 !important;
        }
        .gr-block, .gr-panel, .gr-box, .gr-row, .gr-column {
            background-color: #ffffff !important;
        }
        input, textarea {
            background-color: #ffffff !important;
            color: #000000 !important;
        }
        .dark {
            background-color: #ffffff !important;
        }
    </style>
    """)

    # Header
    gr.Markdown("""
    <h1 style='text-align:center; margin-bottom:0px; color:#FF6F00;'>📈 FinBERT-Financial Sentiment Dashboard</h1>
    <p style='text-align:center; font-size:16px; color:#FF6F00;'>
    Analyze market news, earnings reports, and stock commentary using your fine-tuned FinBERT model.
    </p>
    <br>
    """)

    with gr.Row():

        # Sidebar
        with gr.Column(scale=1, min_width=200):
             gr.Markdown("""
        <div style='padding: 20px; border-radius: 12px; background: #FFD594 !important;'>
            <h3 style='color:#FF6F00 !important;'>📂 Navigation</h3>
            <ul style='font-size:16px; line-height:1.8; color:#000000 !important; list-style:none; padding-left:10px;'>
                <li style='color:#000000 !important;'>🔍 Sentiment Analysis</li>
                <li style='color:#000000 !important;'>📊 Probability Breakdown</li>
                <li style='color:#000000 !important;'>📜 Model Information</li>
                <li style='color:#000000 !important;'>⚙ Settings</li>
            </ul>
        </div>
        """)


        # Main content
        with gr.Column(scale=4):
            gr.Markdown("""
    <h3 style='color:#000000 !important; font-weight:700 !important; margin-bottom:10px;'>
        🔍 Enter Financial Text
    </h3>
    """)
                
            user_input = gr.Textbox(
                label="Text Input",
                placeholder="Example: The stock price is rising today....",
                lines=4
            )
            analyze_btn = gr.Button("Analyze Sentiment", variant="primary")

            gr.Markdown("""
    <h3 style='color:#000000 !important; font-weight:700 !important; margin-top:20px;'>
        📊 Results
    </h3>
    """)
                
            with gr.Row():
                # Prediction 
                with gr.Column():
                    predicted_label = gr.Textbox(label="Predicted Sentiment")
                # Probability 
                with gr.Column():
                    prob_scores = gr.Label(label="Probability Scores")

            analyze_btn.click(
                classify_sentiment,
                inputs=user_input,
                outputs=[predicted_label, prob_scores]
            )

            # Footer 
            gr.Markdown(f""" <br><hr> <p style='text-align:center; color:#FF6F00;'>
            Built with ❤️ using Gradio & FinBERT | PyTorch {torch.__version__}  
            </p> 
            """)

     
# Launch 
demo.launch(theme=gr.themes.Soft(primary_hue="orange"), share = True)


* Running on local URL:  http://127.0.0.1:7860

Could not create share link. Please check your internet connection or our status page: https://status.gradio.app.


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.
